# micrograd: backpropagation from scratch

In module 02 you called `loss.backward()` and gradients appeared. Here you build that machinery yourself in about 100 lines of plain Python, with no PyTorch and no NumPy. Then you'll use it to train a small neural network.

This follows Andrej Karpathy's video "The spelled-out intro to neural networks and backpropagation: building micrograd." Watch it alongside this notebook. Pause the video, implement each piece here yourself, then compare.

If you only take one idea away: a neural network is a big math expression, and backpropagation is the chain rule applied to that expression, from the output backward.

## 1. What a derivative is, numerically

The derivative of `f` at `x` is how much `f` changes when you nudge `x` by a tiny amount `h`, divided by `h`. No calculus rules needed, just evaluate twice.

In [ ]:
def f(x):
    return 3*x**2 - 4*x + 5

h = 1e-6
x = 3.0
print("numerical slope at x=3:", (f(x + h) - f(x)) / h)
print("calculus says 6x - 4 =", 6*x - 4)

This works for any function, but it's far too slow for a network with millions of weights (one extra forward pass per weight) and it's imprecise. Backprop gets every derivative exactly in one backward pass. We'll keep the numerical version around as a checker.

## 2. A Value that remembers how it was made

The core object wraps a number, and records which values produced it and with what operation. Chain operations together and you get a graph of the whole computation.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op="", label=""):
        self.data = data
        self.grad = 0.0              # d(output)/d(this value), filled in by backprop
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value({self.label}={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        return Value(self.data + other.data, (self, other), "+")

    def __mul__(self, other):
        return Value(self.data * other.data, (self, other), "*")


a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")
e = a * b;  e.label = "e"
d = e + c;  d.label = "d"
f_ = Value(-2.0, label="f")
L = d * f_; L.label = "L"

print(L)
print("L was made by", L._op, "from", L._prev)

The graph is: `L = d * f`, `d = e + c`, `e = a * b`. Forward pass: `e = -6`, `d = 4`, `L = -8`.

## 3. Backprop by hand

Goal: `dL/dx` for every value. Start at the output and walk backward, using two local rules:

- For `z = x + y`, the gradient passes through unchanged: `dz/dx = 1`. Addition routes the gradient to both inputs.
- For `z = x * y`, each input's gradient is the other input: `dz/dx = y`. Multiplication swaps them.

The chain rule ties it together: `dL/dx = dL/dz * dz/dx`. The gradient arriving from above times the local derivative.

Work it out on paper before reading on:

| Value | Gradient | Why |
|---|---|---|
| L | 1 | dL/dL |
| d | -2 | L = d·f, so dL/dd = f = -2 |
| f | 4 | dL/df = d = 4 |
| c | -2 | d = e + c, plus routes it: -2 · 1 |
| e | -2 | same |
| a | 6 | e = a·b: dL/da = dL/de · b = -2 · -3 |
| b | -4 | dL/db = dL/de · a = -2 · 2 |

Now check every one numerically.

In [ ]:
def forward(a=2.0, b=-3.0, c=10.0, f=-2.0):
    e = a * b
    d = e + c
    return d * f

h = 1e-6
base = forward()
for name in ["a", "b", "c", "f"]:
    nudged = forward(**{name: {"a": 2.0, "b": -3.0, "c": 10.0, "f": -2.0}[name] + h})
    print(f"dL/d{name} ≈ {(nudged - base) / h:.4f}")

## 4. Automating it

Each operation knows its own local derivative. So when an operation creates a new Value, it also attaches a small function, `_backward`, that pushes the output's gradient into its inputs. Then `backward()` on the final output:

1. Sorts the graph so every node comes after everything it depends on (a topological sort).
2. Sets the output's gradient to 1.
3. Walks the nodes in reverse order, calling each one's `_backward`.

Gradients use `+=`, not `=`. If a value is used in two places, gradient flows back from both, and the contributions add (the multivariable chain rule). Forgetting this is a real, classic bug.

In [ ]:
import math

class Value:
    def __init__(self, data, _children=(), _op="", label=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, k):
        assert isinstance(k, (int, float)), "only constant powers"
        out = Value(self.data ** k, (self,), f"**{k}")
        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")
        def _backward():
            self.grad += (1 - t**2) * out.grad      # d/dx tanh(x) = 1 - tanh(x)^2
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")
        def _backward():
            self.grad += out.data * out.grad        # d/dx e^x = e^x
        out._backward = _backward
        return out

    # Everything else is built from the operations above.
    def __neg__(self):            return self * -1
    def __sub__(self, other):     return self + (-other)
    def __truediv__(self, other): return self * other**-1
    def __radd__(self, other):    return self + other       # lets you write 2 + Value
    def __rmul__(self, other):    return self * other       # lets you write 2 * Value
    def __rsub__(self, other):    return (-self) + other
    def __rtruediv__(self, other): return Value(other) * self**-1

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

In [ ]:
a = Value(2.0); b = Value(-3.0); c = Value(10.0); f_ = Value(-2.0)
L = (a * b + c) * f_
L.backward()
for name, v in [("a", a), ("b", b), ("c", c), ("f", f_), ("L", L)]:
    print(name, v)

Same numbers as your hand calculation. Now the accumulation case: `b = a + a` should give `db/da = 2`.

In [ ]:
a = Value(3.0)
b = a + a
b.backward()
print("db/da =", a.grad, "(would be 1.0 if we'd used = instead of +=)")

## 5. One neuron, checked against PyTorch

A neuron takes inputs `x`, multiplies each by a weight `w`, adds a bias `b`, and squashes the result through an activation like `tanh`: `out = tanh(w1*x1 + w2*x2 + b)`.

The same neuron in micrograd and in PyTorch should give identical gradients.

In [ ]:
x1, x2 = Value(2.0), Value(0.0)
w1, w2 = Value(-3.0), Value(1.0)
b = Value(6.8813735870195432)
o = (x1*w1 + x2*w2 + b).tanh()
o.backward()
print("micrograd:", "o =", round(o.data, 4), "| grads w1, w2, x1, x2:",
      [round(v.grad, 4) for v in (w1, w2, x1, x2)])

import torch
tx1 = torch.tensor(2.0, dtype=torch.float64, requires_grad=True)
tx2 = torch.tensor(0.0, dtype=torch.float64, requires_grad=True)
tw1 = torch.tensor(-3.0, dtype=torch.float64, requires_grad=True)
tw2 = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)
tb = torch.tensor(6.8813735870195432, dtype=torch.float64, requires_grad=True)
to = torch.tanh(tx1*tw1 + tx2*tw2 + tb)
to.backward()
print("pytorch:  ", "o =", round(to.item(), 4), "| grads w1, w2, x1, x2:",
      [round(t.grad.item(), 4) for t in (tw1, tw2, tx1, tx2)])

Notice `w2` has a gradient of 0. Its input `x2` is 0, so changing `w2` can't affect the output. PyTorch's autograd is your class scaled up: tensors instead of single numbers, many more operations, and kernels that run on GPUs.

## 6. A neural network library in 30 lines

In [ ]:
import random

class Neuron:
    def __init__(self, n_in, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_in)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, n_in, n_out, **kwargs):
        self.neurons = [Neuron(n_in, **kwargs) for _ in range(n_out)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, n_in, n_outs):
        sizes = [n_in] + n_outs
        # every layer gets tanh except the last, which stays linear
        self.layers = [Layer(sizes[i], sizes[i+1], nonlin=(i != len(n_outs) - 1)) for i in range(len(n_outs))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

random.seed(1337)
model = MLP(3, [4, 4, 1])     # 3 inputs -> 4 -> 4 -> 1 output
print("parameters:", len(model.parameters()))
print("output on one input:", model([2.0, 3.0, -1.0]))

## 7. Training it

Four examples, each with 3 features and a target of +1 or -1. The network starts random. You'll run the same five-step loop from module 02, except now you built every piece of it.

In [ ]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

random.seed(1337)
model = MLP(3, [4, 4, 1])
lr = 0.05

for step in range(100):
    preds = [model(x) for x in xs]                              # 1. forward
    loss = sum((p - y)**2 for p, y in zip(preds, ys))           # 2. loss
    for p in model.parameters():                                # 3. zero grads
        p.grad = 0.0
    loss.backward()                                             # 4. backward
    for p in model.parameters():                                # 5. update
        p.data -= lr * p.grad
    if step % 10 == 0:
        print(f"step {step:3}  loss {loss.data:.5f}")

print("\npredictions:", [round(p.data, 3) for p in preds])
print("targets:    ", ys)

That's a neural network learning, and you wrote every line of the math. Everything bigger is the same idea with more efficient engineering:

- Tensors batch thousands of these scalar operations into one GPU kernel.
- Better optimizers (Adam) replace `p.data -= lr * p.grad`.
- Better loss functions (cross-entropy) replace squared error for classification.
- Better architectures (transformers) replace the MLP.

---
# Exercises

**Exercise 1: Add ReLU.** ReLU is `max(0, x)` and is the most common activation in modern networks. Add a `relu()` method to `Value` with the correct `_backward` (what's the derivative when `x > 0`? when `x < 0`?). Verify against `torch.relu` for an input of `-2.0` and `3.0`.

**Exercise 2: A gradient checker.** Write `grad_check(fn, values, h=1e-6)` that takes a function building an expression from a list of floats, runs your backprop, computes numerical gradients by nudging each input, and prints both side by side. Test it on `lambda a, b, c: ((a * b).tanh() + c**2) / (a + 3)`.

**Exercise 3: Break it on purpose.** Rerun the training loop with the "zero grads" step deleted. Watch the loss. Explain what's happening to the gradients each step.

**Exercise 4: Learning rate.** Train fresh models for 50 steps with `lr` set to 0.001, 0.05, and 0.5. Which barely moves, which works, and which blows up? Learning rate is the first thing you tune on any real model.

---
# Solutions

In [ ]:
# Exercise 1
def relu(self):
    out = Value(max(0.0, self.data), (self,), "relu")
    def _backward():
        self.grad += (1.0 if self.data > 0 else 0.0) * out.grad   # slope 1 if positive, 0 otherwise
    out._backward = _backward
    return out

Value.relu = relu   # attach the method to the existing class

for x in [-2.0, 3.0]:
    v = Value(x); out = v.relu(); out.backward()
    t = torch.tensor(x, requires_grad=True); torch.relu(t).backward()
    print(f"x={x}: micrograd grad {v.grad}, pytorch grad {t.grad.item()}")

In [ ]:
# Exercise 2
def grad_check(fn, values, h=1e-6):
    vals = [Value(v) for v in values]
    out = fn(*vals)
    out.backward()
    base = fn(*[Value(v) for v in values]).data
    for i, v in enumerate(values):
        nudged = list(values); nudged[i] = v + h
        numeric = (fn(*[Value(n) for n in nudged]).data - base) / h
        print(f"input {i}: backprop {vals[i].grad: .6f}   numeric {numeric: .6f}")

grad_check(lambda a, b, c: ((a * b).tanh() + c**2) / (a + 3), [0.5, -1.2, 2.0])

In [ ]:
# Exercise 3: without zeroing, each step's gradient is added on top of every previous step's.
# The effective update grows every step, so training overshoots and the loss stops falling smoothly
# (it may oscillate or blow up). This is exactly why PyTorch makes you call optimizer.zero_grad().
random.seed(1337)
model = MLP(3, [4, 4, 1])
for step in range(30):
    preds = [model(x) for x in xs]
    loss = sum((p - y)**2 for p, y in zip(preds, ys))
    loss.backward()                      # no zeroing
    for p in model.parameters():
        p.data -= 0.05 * p.grad
    if step % 3 == 0:
        print(f"step {step:3}  loss {loss.data:.5f}")

In [ ]:
# Exercise 4
# 0.001 learns slowly, 0.05 converges fast, 0.5 overshoots so badly the loss explodes (diverges).
for lr in [0.001, 0.05, 0.5]:
    random.seed(1337)
    model = MLP(3, [4, 4, 1])
    history = []
    for step in range(50):
        preds = [model(x) for x in xs]
        loss = sum((p - y)**2 for p, y in zip(preds, ys))
        for p in model.parameters():
            p.grad = 0.0
        loss.backward()
        for p in model.parameters():
            p.data -= lr * p.grad
        history.append(loss.data)
    print(f"lr={lr:<6} loss at steps 0/10/25/49: " + ", ".join(f"{history[i]:.4g}" for i in (0, 10, 25, 49)))